# Indic Synthetic-Speech Pipeline — Colab (T4) stage-by-stage runbook

Validate **one stage at a time** on the GPU, then commit and move on.

**The loop (per stage):** edit locally in VSCode → `git push` → here run `!git pull` → run the stage → inspect the manifest + listen to audio → fix & repeat → commit `"stage N validated"`.

**Why outputs go to Drive:** each stage reads the previous stage's manifest, and Colab runtimes disconnect. With `out_dir` on Drive (see `config.colab.yaml`) the manifests + audio survive restarts, so the per-stage sessions below can run in *separate* runtimes.

**transformers conflict:** IndicF5 pins `==4.49.0` but Gemma-3 needs `>=4.50`. We handle it by giving each session its own install and **restarting the runtime** between Sessions 1→2→3.

> ⚠️ Never hardcode an HF token in a committed cell. We read it via `getpass`. If you ever leaked one, revoke it at https://huggingface.co/settings/tokens .

## 0. One-time setup (run at the start of every session)
Mount Drive, clone-or-pull the repo, set the HF token. Set `REPO_URL` to your GitHub repo.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/<you>/synthetic-data-pipeline.git'  # <-- set this
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only
os.makedirs('/content/drive/MyDrive/indic_synth/out', exist_ok=True)

In [ ]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token: ')
login(os.environ['HF_TOKEN'])

---
## Session 1 — §4.2 acquisition + §4.3 audio engineering
No transformers model; CPU is fine. Run §4.2, inspect, then §4.3, inspect.

In [ ]:
!pip install -q pyarrow pandas fsspec huggingface_hub datasets soundfile soxr librosa pyyaml
!pip install -q -e .

In [ ]:
# §4.2 — pull the gender-balanced reference voice bank from Kathbath
!python scripts/run.py --config config.colab.yaml --stages data_acquisition

In [ ]:
# inspect + listen to a reference clip
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/reference_manifest.jsonl --n 3
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/ref_audio/*'))[:2]:
    print(w); display(Audio(w))

In [ ]:
# §4.3 — decode / resample to 24 kHz mono / normalize
!python scripts/run.py --config config.colab.yaml --stages audio_engineering
!python scripts/inspect_manifest.py $OUT/prepared_manifest.jsonl --n 3

In [ ]:
# verify a prepared clip really is 24 kHz mono, and listen
import soundfile as sf, glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/prepared_audio/*.wav'))[:2]:
    i = sf.info(w); print(w, i.samplerate, 'Hz', i.channels, 'ch'); display(Audio(w))

**✅ If §4.2/§4.3 look right** (20 speakers, gender-balanced, 24 kHz mono refs): commit `"stage 4.2/4.3 validated"` from VSCode.

**Watch-outs:** gated 403 → accept Kathbath terms on the hub. If no files are found, the real HF layout / language-folder names may differ from the `datasets/ai4bharat/Kathbath/<lang>/valid-*.parquet` glob in `src/indic_synth/data_acquisition/hf_io.py` — adjust `languages` in `config.colab.yaml` or the glob. Kathbath clips may be **m4a** (decoded via librosa/ffmpeg in `audio_engineering/prepare.py`).

---
## Session 2 — §4.4 sentence generation (Gemma-3)
**Restart the runtime first** (Runtime → Restart), then re-run section 0, then this.
Needs `transformers>=4.50` + a GPU.

In [ ]:
!pip install -q "transformers>=4.50" accelerate bitsandbytes sentence-transformers fasttext-wheel indic-num2words pyyaml
!pip install -q -e .

In [ ]:
# §4.4 — grid-balanced, validated Indic sentences (checkpointed; safe to re-run)
!python scripts/run.py --config config.colab.yaml --stages sentence_generation

In [ ]:
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/sentences.jsonl --n 5
import json
print(open(f'{OUT}/sentences_summary.json').read())

**✅ Check:** per-language / per-type / per-topic counts are balanced, QC yield is reasonable, and sampled sentences read fluently. Commit `"stage 4.4 validated"`.

**Watch-outs:** accept the **Gemma-3 license**; `bitsandbytes` 4-bit needs the GPU; `lid.176.bin` (~126 MB) downloads into `out_dir`; confirm `processor.apply_chat_template(...)` works for the installed transformers (`sentence_generation/models.py`).

---
## Session 3 — §4.5 TTS (IndicF5) + §4.6 QC
**Restart the runtime first**, re-run section 0, then this. IndicF5 from source + `transformers==4.49.0`.

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml
!pip install -q -e .

In [ ]:
# §4.5 — IndicF5 speaks each sentence in a same-language speaker's voice
!python scripts/run.py --config config.colab.yaml --stages tts_generation
OUT = '/content/drive/MyDrive/indic_synth/out'
!python scripts/inspect_manifest.py $OUT/tts_manifest.jsonl --n 3

In [ ]:
# listen to a few synthesized utterances
import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'{OUT}/tts_audio/*.wav'))[:4]:
    print(w); display(Audio(w))

In [ ]:
# §4.6 — three QC gates -> final dataset_manifest.jsonl
!python scripts/run.py --config config.colab.yaml --stages quality_control
!python scripts/inspect_manifest.py $OUT/dataset_manifest.jsonl --n 3
import json
print(open(f'{OUT}/qc_summary.json').read())

In [ ]:
# sanity-check thresholds: listen to a couple of QC failures
from indic_synth.common.manifest import read_jsonl
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:3]
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))

**✅ Done** when `dataset_manifest.jsonl` holds ~1000 validated utterances across 2 languages / 20 speakers with a sensible QC pass rate. Commit `"stage 4.5/4.6 validated"`.

**Watch-outs:** IndicF5 git build + `trust_remote_code`; confirm the `tts_model(text, ref_audio_path=..., ref_text=...)` call + int16→float32 handling in `tts_generation/models.py`; T4 VRAM for IndicF5. For QC: the `indic-conformer` `model(wav, lang_code, 'ctc')` signature and the **lang_code it expects** (map `hi`/`ml` if needed); speechbrain ECAPA load. If conformer needs a different transformers than 4.49, run §4.6 in its own session — the Drive manifests make that safe.